# Gym-Duckietown: lane following, Pure Pursuit, and falsifying failure modes

This notebook uses the **maintained fork** [**ai-vnv/gym-duckietown**](https://github.com/ai-vnv/gym-duckietown) (`daffy` branch): modern NumPy/Pyglet, PWM/NumPy integration fix, and macOS-friendly OpenGL. Upstream: [duckietown/gym-duckietown](https://github.com/duckietown/gym-duckietown).

**Install (local clone of the fork):**

```bash
git clone -b daffy https://github.com/ai-vnv/gym-duckietown.git
cd gym-duckietown
python3.10 -m venv .venv && source .venv/bin/activate
pip install -e .
pip install jupyter imageio imageio-ffmpeg
jupyter notebook notebooks/gym_duckietown_pedagogy.ipynb
```

**Staying current with Duckietown:** from your clone, `git remote add upstream https://github.com/duckietown/gym-duckietown.git` (once), then periodically:

```bash
git fetch upstream && git merge upstream/daffy
git push origin daffy
```

Full **fork vs upstream** notes: [README.md](https://github.com/ai-vnv/gym-duckietown/blob/daffy/README.md) (sections **Fork note**, not original Duckietown copy).

The next cell sets up the path on **Google Colab** (clone fork + `pip install`) or **local** (repo root detection).


In [ ]:
import os
import sys
import subprocess
import pathlib

# Fork maintained by ai-vnv lab (includes patches; branch daffy)
FORK_URL = "https://github.com/ai-vnv/gym-duckietown.git"
BRANCH = "daffy"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()

if IN_COLAB:
    REPO = pathlib.Path("/content/gym-duckietown")
    if not (REPO / "setup.py").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, FORK_URL, str(REPO)],
            check=True,
        )
    # Colab Linux: virtual framebuffer helps some Pyglet/OpenGL setups
    subprocess.run(
        "apt-get update -qq && apt-get install -qq -y xvfb freeglut3-dev >/dev/null",
        shell=True,
        check=False,
    )
    if not os.environ.get("DISPLAY"):
        subprocess.Popen(
            ["Xvfb", ":99", "-screen", "0", "1024x768x24", "-ac", "+extension", "GLX", "+render", "-noreset"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        os.environ["DISPLAY"] = ":99"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "imageio", "imageio-ffmpeg"],
        check=True,
    )
else:
    here = pathlib.Path.cwd().resolve()
    REPO = here.parent if here.name == "notebooks" else here
    if not (REPO / "setup.py").is_file():
        raise RuntimeError(
            "Run Jupyter from the repo root or from notebooks/ so the fork root is found. "
            f"Expected setup.py at {REPO / 'setup.py'}"
        )

REPO = str(REPO)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "learning", "imitation", "iil-dagger"))
print("REPO =", REPO, "| IN_COLAB =", IN_COLAB)


In [ ]:
import gym
import numpy as np

import gym_duckietown  # registers envs; PWM/NumPy patch from fork
from IPython.display import Video, display
from teacher.pure_pursuit_policy import PurePursuitPolicy


### Pure Pursuit baseline

The imitation-learning stack uses a **Pure Pursuit** controller: it tracks a lookahead point on the lane centerline (from simulator state, not from pixels). Steering is proportional to lateral error; velocity is reduced in curves. Examples scale the steering command with **`omega_gain`** (often ~7).

**Falsification:** hypothesis *H* = “with default-ish parameters, the car stays in a valid pose until `max_steps`.” Search over `(seed, omega_gain, env_id)` until `done` with an “invalid pose” message.


In [ ]:
def rollout_pp(env_id: str, seed: int, omega_gain: float, ref_v: float = 0.7, fd: float = 0.24, max_steps: int = 500):
    env = gym.make(env_id, disable_env_checker=True)
    env.seed(seed)
    obs = env.reset()
    pol = PurePursuitPolicy(env.unwrapped, ref_velocity=ref_v, following_distance=fd)
    total_r = 0.0
    last_msg = ""
    for t in range(max_steps):
        raw = pol.predict(obs)
        a = np.array([float(raw[0]), float(raw[1]) * omega_gain], dtype=np.float32)
        obs, r, done, info = env.step(a)
        total_r += r
        last_msg = info.get("Simulator", {}).get("msg", "")
        if done:
            env.close()
            return t + 1, total_r, last_msg
    env.close()
    return max_steps, total_r, last_msg


def falsify_invalid_pose(env_id, seeds=range(0, 15), omegas=(7.0, 12.0, 18.0)):
    for og in omegas:
        for s in seeds:
            steps, ret, msg = rollout_pp(env_id, s, og)
            if "invalid" in msg.lower():
                return {"seed": s, "omega_gain": og, "steps": steps, "msg": msg}
    return None

print("small_loop", falsify_invalid_pose("Duckietown-small_loop-v0"))
print("loop_obstacles", falsify_invalid_pose("Duckietown-loop_obstacles-v0", seeds=range(0, 8)))

### Two recorded failure modes

1. **High steering gain** on `Duckietown-small_loop-v0`: large `omega_gain` overshoots lane tracking and leaves the drivable mesh (`invalid pose`).
2. **`loop_obstacles` map** with default gain: Pure Pursuit follows the lane curve in pose space but does not “see” obstacles; the vehicle ends on a **floor** tile beside the road.

Run `python scripts/record_pp_failures.py` from the repo root to regenerate MP4s in `recordings/`.


In [ ]:
rec = os.path.join(REPO, "recordings")
v1 = os.path.join(rec, "failure_high_omega_gain.mp4")
v2 = os.path.join(rec, "failure_obstacles_map.mp4")
for v in (v1, v2):
    if os.path.isfile(v):
        display(Video(v, embed=True, width=640))
    else:
        print("Missing:", v)